In [1]:
pip install peft transformers accelerate bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install flask

Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import random
import numpy as np
import pandas as pd
import nltk
import torch

from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
lemmatizer = WordNetLemmatizer()


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mabua\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mabua\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mabua\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [4]:
with open('intents.json', 'r') as file:
    intents = json.load(file)


In [5]:

words=[]
classes=[]
documents=[]

for intent in intents['intents']:
    tag = intent['tag']
    if tag not in classes:
        classes.append(tag)
    for pattern in intent['patterns']:
        w = nltk.word_tokenize(pattern.lower())
        words.extend(w)
        documents.append((w, tag))

#here we normalizing the vocabs
words = [lemmatizer.lemmatize(w) for w in words if w.isalnum()]
words = sorted(list(set(words)))
classes = sorted(list(set(classes)))

# bag of words training data
training = []
for token_list, tag in documents:
    bag = [1 if lemmatizer.lemmatize(w) in [lemmatizer.lemmatize(x) for x in token_list] else 0 for w in words]
    label = classes.index(tag)
    training.append((bag, label))

random.shuffle(training)
X = np.array([t[0] for t in training])
y = np.array([t[1] for t in training])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [6]:

#here we are using a small conversational LLM for real “vanilla” prompting (no fine-tuning).

prompt_name = "microsoft/DialoGPT-medium"


prompt_tokenizer = AutoTokenizer.from_pretrained(prompt_name)
prompt_model = AutoModelForCausalLM.from_pretrained(prompt_name)


if prompt_tokenizer.pad_token is None:
    prompt_tokenizer.pad_token = prompt_tokenizer.eos_token

prompt_pipe = pipeline(
    "text-generation",
    model=prompt_model,
    tokenizer=prompt_tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

def respond_vanilla(question: str) -> str:
    system = "You are a helpful college enquiry chatbot. Answer briefly and accurately."
    prompt = f"{system}\nQuestion: {question}\nAnswer:"
    response = prompt_pipe(
        prompt,
        max_new_tokens=60,
        temperature=0.3,
        do_sample=True,
        pad_token_id=prompt_tokenizer.eos_token_id,
        num_return_sequences=1
    )[0]['generated_text']

    if "Answer:" in response:
        return response.split("Answer:")[-1].strip()
    return response.strip()



Device set to use cpu


In [7]:
train_texts = []

for intent in intents["intents"]:
    for p in intent["patterns"]:
        for r in intent["responses"]:
            train_texts.append(f"Question: {p}\nAnswer: {r}")


In [8]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = prompt_tokenizer

def tokenize_fn(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


dataset = Dataset.from_dict({"text": train_texts})
dataset = dataset.map(tokenize_fn, batched=True)

lora_train_dataset = dataset
lora_eval_dataset = dataset.select(range(10))


Map:   0%|          | 0/453 [00:00<?, ? examples/s]

In [9]:
# this is  (Optional) Prepare LoRA/PEFT Fine-Tuning


from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
     r=8,
     lora_alpha=16,
     target_modules=["c_attn", "c_proj"] if hasattr(prompt_model, "transformer") else None,
     lora_dropout=0.05,
     bias="none",
     task_type="CAUSAL_LM"
 )

lora_model = get_peft_model(prompt_model, lora_config)
lora_model.print_trainable_parameters()


from transformers import TrainingArguments, Trainer
train_args = TrainingArguments(
    output_dir="./lora_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=True if torch.cuda.is_available() else False,
    logging_steps=10,
    save_steps=100,
    max_steps=20, 
)
trainer = Trainer(
    model=lora_model,
    args=train_args,
    train_dataset=lora_train_dataset,
    eval_dataset=lora_eval_dataset 
 )
trainer.train()

d:\anaconda\Lib\site-packages\peft\tuners\lora\layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 2,162,688 || all params: 356,985,856 || trainable%: 0.6058



d:\anaconda\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,5.743600
20,2.294600


TrainOutput(global_step=20, training_loss=4.019111633300781, metrics={'train_runtime': 175.4888, 'train_samples_per_second': 0.912, 'train_steps_per_second': 0.114, 'total_flos': 37413778882560.0, 'train_loss': 4.019111633300781, 'epoch': 0.3524229074889868})

In [10]:

#Fine-Tuning (NN Classifier on intents)

model = Sequential([
    Dense(128, input_shape=(len(words),), activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(len(classes), activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

_ = model.fit(X_train, y_train, epochs=80, batch_size=8,
              validation_data=(X_test, y_test), verbose=0)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f" accuracy : {acc:.3f}")

d:\anaconda\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


 accuracy : 0.742


In [11]:
 #Helping Functions + TF-IDF Prompting

def clean_up_sentence(sentence):
    tokens = nltk.word_tokenize(sentence.lower())
    return [lemmatizer.lemmatize(w) for w in tokens if w.isalnum()]

def bow_vector(sentence, words_list):
    sentence_words = clean_up_sentence(sentence)
    return np.array([1 if w in sentence_words else 0 for w in words_list])

# TF-IDF over patterns (prompting simulator // backup)
pattern_texts, pattern_tags = [], []
for intent in intents['intents']:
    for p in intent['patterns']:
        pattern_texts.append(p)
        pattern_tags.append(intent['tag'])

tfidf_vectorizer = TfidfVectorizer().fit(pattern_texts)
pattern_vecs = tfidf_vectorizer.transform(pattern_texts)

def respond_tfidf(query: str):
    " TF-IDF PROMPTING  finds closest pattern and returns a canned response of its tag"
    qv = tfidf_vectorizer.transform([query])
    sims = cosine_similarity(qv, pattern_vecs)[0]
    best_idx = sims.argmax()
    best_tag = pattern_tags[best_idx]
    for intent in intents['intents']:
        if intent['tag'] == best_tag:
            return random.choice(intent['responses']), best_tag, float(sims[best_idx])
    return "No  answer found", None, 0.0

def predict_nn_tag(query: str):
    bow = bow_vector(query, words).reshape(1, -1)
    probs = model.predict(bow, verbose=0)[0]
    idx = int(np.argmax(probs))
    return classes[idx], float(probs[idx])

def respond_finetuned(query: str, threshold: float = 0.65):
    tag, prob = predict_nn_tag(query)
    if prob >= threshold:
        for intent in intents['intents']:
            if intent['tag'] == tag:
                return random.choice(intent['responses']), tag, prob
   
    unknown_responses = [
        "Sorry, I didn’t quite understand that",
        "I’m here to answer questions about the college. Could you rephrase?",
        "I might have missed that—can you try again?"
    ]
    return random.choice(unknown_responses), "unknown", prob

In [12]:
#checking four modes
#we can add mulitple things or tests it's optional
tests = [
    "How do I apply for admission?",
    "Does the college have a library?",
    "shut up",
    "dikhandkhpad"
]
rows = []
for q in tests:
    v = respond_vanilla(q)
    t_ans, t_tag, t_sim = respond_tfidf(q)
    nn_tag, nn_prob = predict_nn_tag(q)
    f_ans, f_tag, f_prob = respond_finetuned(q, threshold=0.65)
    rows.append({
        "Query": q,
        "Vanilla": v[:120] + ("....." if len(v) > 120 else ""),
        "TF-IDF  Tag": t_tag, "TF-IDF  Sim": round(t_sim, 3),
        "NN  Tag": nn_tag, "NN  Prob": round(nn_prob, 3),
        "Fine-tuned  Tag": f_tag, "Fine-tuned  Prob": round(f_prob, 3),
        "Fine-tuned  Response": f_ans
    })
pd.DataFrame(rows)

,Query,Vanilla,TF-IDF Tag,TF-IDF Sim,NN Tag,NN Prob,Fine-tuned Tag,Fine-tuned Prob,Fine-tuned Response
0,How do I apply for admission?,How do I get in?,document,0.601,document,0.922,document,0.922,"you will be asked to bring your 10th, 12th mar..."
1,Does the college have a library?,Yes,library,0.878,library,1.000,library,1.000,There is one huge and spacious library.timings...
2,shut up,shut up,swear,1.000,swear,0.996,swear,0.996,Maintaining decency would be appreciated
3,dikhandkhpad,dikhandkhpad,greeting,0.000,greeting,0.414,unknown,0.414,"Sorry, I didn’t quite understand that"


In [13]:
#building +30 questions
test_prompts = []

for intent in intents['intents']:
    for p in intent['patterns'][:2]:
        test_prompts.append((p, intent['tag']))

extras = [
    ("How do I apply for admission?", "admission"),
    ("Does the college have a library?", "library"),
    ("What is the college timing?", "hours"),
    ("Who created you?", "creator"),
    ("Do you have a canteen?", "canteen"),
    ("Where is the college located?", "location"),
    ("Is there a vacation this week?", "vacations"),
    ("What scholarships are available?", "scholarship"),
]
test_prompts.extend(extras)
test_prompts = test_prompts[:30]

pd.DataFrame(test_prompts, columns=["Question", "Expected Tag"])

,Question,Expected Tag
0,Hi,greeting
1,How are you?,greeting
2,cya,goodbye
3,see you,goodbye
4,what is the name of your developers,creator
5,what is the name of your creators,creator
6,name,name
7,your name,name
8,timing of college,hours
9,what is college timing,hours


In [14]:
tag_refs = {}
for intent in intents['intents']:
    tag_refs[intent['tag']] = " ".join(intent['responses'])

tag_names = sorted(tag_refs.keys())
tag_corpus = [tag_refs[t] for t in tag_names]
judge_vectorizer = TfidfVectorizer().fit(tag_corpus)
tag_ref_vecs = judge_vectorizer.transform(tag_corpus)

def map_answer_to_tag(answer_text):
    if not answer_text or answer_text.strip() == "":
        return None, 0.0
    av = judge_vectorizer.transform([answer_text])
    sims = cosine_similarity(av, tag_ref_vecs)[0]
    idx = sims.argmax()
    return tag_names[idx], float(sims[idx])

def vanilla_label(sim_score, true_tag, pred_tag):
   
    if pred_tag == true_tag and sim_score >= 0.35:
        return " Correct"
    elif pred_tag == true_tag and sim_score >= 0.20:
        return " Partial"
    else:
        return " Wrong"

vanilla_rows = []
for q, true_tag in test_prompts:
    ans = respond_vanilla(q)
    pred_tag, sim = map_answer_to_tag(ans)
   
    verdict = vanilla_label(sim, true_tag, pred_tag if pred_tag is not None else "")
    vanilla_rows.append({
        "Query": q,
        "True Tag": true_tag,
        "Vanilla Answer": ans,
        "Vanilla Pred Tag": pred_tag,
        "Vanilla Similarity": round(sim, 3),
        "Vanilla Correctness": verdict
    })

df_vanilla = pd.DataFrame(vanilla_rows)
df_vanilla[["Query","True Tag","Vanilla Pred Tag","Vanilla Similarity","Vanilla Correctness"]]

,Query,True Tag,Vanilla Pred Tag,Vanilla Similarity,Vanilla Correctness
0,Hi,greeting,admission,0.000,Wrong
1,How are you?,greeting,greeting,0.329,Partial
2,cya,goodbye,None,0.000,Wrong
3,see you,goodbye,goodbye,0.378,Correct
4,what is the name of your developers,creator,hod,0.307,Wrong
5,what is the name of your creators,creator,floors,0.335,Wrong
6,name,name,admission,0.000,Wrong
7,your name,name,vacation,0.393,Wrong
8,timing of college,hours,creator,0.378,Wrong
9,what is college timing,hours,None,0.000,Wrong


In [15]:
rows = []
tfidf_correct = 0
nn_correct = 0

for q, true_tag in test_prompts:

    _, p_tag, _ = respond_tfidf(q)


    nn_tag, nn_prob = predict_nn_tag(q)

    rows.append({
        "Query": q,
        "True Tag": true_tag,
        "TF-IDF  Pred Tag": p_tag,
        "NN → Pred Tag": nn_tag,
        "TF-IDF Correct": "yes" if p_tag == true_tag else "no",
        "NN Correct": "yes" if nn_tag == true_tag else "no"
    })

    tfidf_correct += int(p_tag == true_tag)
    nn_correct += int(nn_tag == true_tag)


df_comp = pd.DataFrame(rows)
df_comp

,Query,True Tag,TF-IDF Pred Tag,NN → Pred Tag,TF-IDF Correct,NN Correct
0,Hi,greeting,greeting,greeting,yes,yes
1,How are you?,greeting,greeting,greeting,yes,yes
2,cya,goodbye,goodbye,goodbye,yes,yes
3,see you,goodbye,goodbye,goodbye,yes,yes
4,what is the name of your developers,creator,creator,creator,yes,yes
5,what is the name of your creators,creator,creator,creator,yes,yes
6,name,name,name,name,yes,yes
7,your name,name,name,name,yes,yes
8,timing of college,hours,hours,hours,yes,yes
9,what is college timing,hours,hours,hours,yes,yes


In [ ]:
    tag_summary = df_comp.groupby('True Tag').agg({
        'TF-IDF Correct': lambda s: sum(v == 'yes' for v in s),
        'NN Correct': lambda s: sum(v == 'yes' for v in s)
    })
    display(tag_summary)




,TF-IDF Correct,NN Correct
True Tag,,
canteen,2,2
course,2,2
creator,2,2
document,2,2
event,2,2
floors,2,2
goodbye,2,2
greeting,2,2
hours,2,2


In [17]:
vanilla_tag_counts = df_vanilla.groupby('True Tag')['Vanilla Correctness'].value_counts().unstack().fillna(0).astype(int)
display(vanilla_tag_counts)

Vanilla Correctness,Correct,Partial,Wrong
True Tag,,,
canteen,0,0,2
course,0,0,2
creator,0,0,2
document,0,0,2
event,0,0,2
floors,0,0,2
goodbye,1,0,1
greeting,0,1,1
hours,0,0,2


In [18]:

# Overall Comparison

tfidf_acc = (df_comp['TF-IDF Correct'] == 'yes').mean()
nn_acc    = (df_comp['NN Correct'] == 'yes').mean()


score_map = {" Correct": 1.0, " Partial": 0.5, " Wrong": 0.0}
vanilla_score = df_vanilla['Vanilla Correctness'].map(score_map).fillna(0.0).mean()

summary_table = pd.DataFrame({
    "Approach": ["Vanilla (LLM)", "TF-IDF Prompting", "Fine-tuned NN (tag)"],
    "Score / Accuracy": [round(vanilla_score, 3), round(tfidf_acc, 3), round(nn_acc, 3)],


})
display(summary_table)



,Approach,Score / Accuracy
0,Vanilla (LLM),0.067
1,TF-IDF Prompting,1.000
2,Fine-tuned NN (tag),1.000


In [ ]:
def chatbot_reply(message):
    ints = predict_class(message, model)
    response = get_response(ints, intents)
    return response


In [20]:
print("\n ask chatbot about college type ' quit' to exit ")

THRESH = 0.65  

unknown_responses = [
    "Sorry, I didn’t quite understand that.",
    "I'm here to talk about the college. Could you rephrase?",
    "I might have missed that—can you try again?"
]

while True:
    msg = input('You: ')
    if msg.lower() in ['quit', 'exit']:
        print('Chatbot: Goodbye!')
        break


    tag, prob = predict_nn_tag(msg)
    resp_text = None

    if prob >= THRESH:

        for intent in intents['intents']:
            if intent['tag'] == tag:
                resp_text = random.choice(intent['responses'])
                break
    else:

        tfidf_resp, tfidf_tag, tfidf_sim = respond_tfidf(msg)
        if tfidf_tag is not None and tfidf_sim >= 0.25:
            resp_text = tfidf_resp

    if not resp_text:
        resp_text = random.choice(unknown_responses)

    print('Chatbot:', resp_text)


 ask chatbot about college type ' quit' to exit 
Chatbot: Hi there, how can I help?
Chatbot: Goodbye!
